In [ ]:
# --- Imports ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import pandas as pd
import numpy as np
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split

# --- Configuration ---
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8  # 360 / 45 degrees bins = 8 classes
BIN_SIZE = 360 // NUM_CLASSES
DATA_DIR = Path('./data') # Update to your path
CSV_PATH = DATA_DIR / 'tags.csv'

# --- Helper Functions ---
def calculate_angle(center, top):
    """Calculates angle between vertical vector and center->top vector"""
    vector = top - center
    vector = vector / (np.linalg.norm(vector) + 1e-6)
    ref_vector = np.array([0, -1]) # Vertical up
    
    # Calculate angle in degrees
    angle = np.degrees(np.arctan2(vector[1], vector[0]) - np.arctan2(ref_vector[1], ref_vector[0]))
    if angle < 0:
        angle += 360
    return angle

def binarize_angle(angle, bin_size):
    return int((angle / bin_size) % (360 // bin_size))

# --- Dataset ---
class ClockRotationDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.file_names = df['crop_file'].unique()

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        group = self.df[self.df['crop_file'] == fname]
        
        # Load Image
        img_path = self.img_dir / fname
        image = Image.open(img_path).convert('RGB')
        
        # Get Keypoints
        try:
            center = group[group['label'] == 'Center'][['x', 'y']].values[0]
            top = group[group['label'] == 'Top'][['x', 'y']].values[0]
            
            # Scale keypoints to original image size (assuming csv has normalized 0-1 or pixel coords)
            # If CSV is normalized:
            # center = center * np.array(image.size)
            # top = top * np.array(image.size)
            
            angle = calculate_angle(center, top)
            label = binarize_angle(angle, BIN_SIZE)
        except IndexError:
            # Handle missing keypoints
            label = 0 

        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

# --- Data Preparation ---
df = pd.read_csv(CSV_PATH)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ClockRotationDataset(train_df, DATA_DIR, transform=transform)
val_ds = ClockRotationDataset(val_df, DATA_DIR, transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# --- Model ---
model = models.efficientnet_b0(pretrained=True)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# --- Training Loop ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")